In [ ]:
# pip install off

# Loading a trained flow and the grid (quadrature) energy for H$_2$

A trained continuous normalizing flow (CNF) defines the electron density
$\rho(\mathbf{x}) = N_e\,\rho_\phi(\mathbf{x})$. During training the OF-DFT energy is a
**Monte-Carlo** average over flow samples; here we **reload a finished run** and evaluate
the same energy by **deterministic quadrature** on an atom-centred (Becke) grid,
$$
E = \int \big(T + V_{\rm ne} + V_{\rm H} + E_{\rm xc}\big)\,d\mathbf{x} \; + \; E_{\rm nn},
$$
which removes the Monte-Carlo noise and gives a clean term-by-term breakdown.

In [ ]:
import json, glob, re
import jax
import jax.numpy as jnp
import jax.random as jrnd
import equinox as eqx

import off
from off.train import setup_molecule, setup_model
from off.utils import get_solver
from off.promolecular.promolecular_dist import ProMolecularDensity
from off import build_energy_functional, grid_energy, grid_energy_from_checkpoint

jax.config.update("jax_enable_x64", True)
print("off version:", off.__version__)

## 1. The trained run

A run directory holds `job_params.json` (the functional and geometry) and
`Checkpoints/checkpoint_*.eqx` (the flow parameters) — the layout written by `off-train`
and `off.training`. Point `run_dir` at one such directory.

In [ ]:
run_dir = "h2_run/bl_1.40"

ckpts = sorted(glob.glob(f"{run_dir}/Checkpoints/checkpoint_*.eqx"),
               key=lambda s: int(re.search(r"checkpoint_(\d+)\.eqx", s).group(1)))
assert ckpts, f"No checkpoints in {run_dir}/Checkpoints/ - train a model first (off-train or off.training)."

with open(f"{run_dir}/job_params.json") as f:
    p = json.load(f)
print("run:", p["mol_name"], "| R =", p["bond_length"], "Bohr | latest checkpoint:", ckpts[-1])

## 2. Reload the flow

Rebuild the molecule and an empty CNF with the **same architecture** recorded in
`job_params.json`, then deserialize the saved parameters into it.

In [ ]:
Ne, atoms, z, coords, mol = setup_molecule(p["mol_name"], p["bond_length"])

skeleton   = setup_model(coords, z, p["hidden_layer"], jrnd.PRNGKey(0))
flow_model = eqx.tree_deserialise_leaves(ckpts[-1], skeleton)
solver     = get_solver(p["solver"])
prior      = ProMolecularDensity(z.ravel(), coords)
print("loaded flow with", p["hidden_layer"], "hidden units")

## 3. Grid (quadrature) energy

Rebuild the `EnergyFunctional` from the recorded functional names, then integrate every
term on the Becke grid. `grid_energy` evaluates $\rho_\phi$ and its score on the grid via
the flow, applies each density functional, and returns the energy breakdown together with
$\int\rho\,d\mathbf{x}$ as a normalization check. `grid_level` trades accuracy for cost.

In [ ]:
functional = build_energy_functional(
    kinetic_name=p["kinetic"], lam=p["lam"], exchange_name=p["exchange"],
    correlation_name=p["correlation"], hartree_name=p["hartree"],
    external_name=p["external"], core_correction_name=p["core_correction"],
)

en = grid_energy(flow_model, prior, solver, coords, z, atoms, Ne, functional,
                 grid_level=3, units="bohr")

for k in ("T", "V_N", "V_H", "E_X", "E_C", "E_CC", "E_NN"):
    print(f"{k:8s} = {en[k]:+.6f} Ha")
print("-" * 28)
print(f"{'E_total':8s} = {en['E_total']:+.6f} Ha")
print(f"Ne (int rho) = {en['Ne_integral']:.4f}   (target {Ne})")

## 4. One call from the run directory

`grid_energy_from_checkpoint` does all of the above — read `job_params.json`, load the
last checkpoint, build the functional, and integrate — in a single call, caching the
result in `energy_summary.json`.

In [ ]:
en = grid_energy_from_checkpoint(run_dir, grid_level=3)
print(f"E_total = {en['E_total']:+.6f} Ha   (epoch {en['epoch']}, R = {en['bond_length']} Bohr)")